# WESAD — 03 · Leakage, Confounding & LOSO Generalization

*Project: When Wearable Stress Models Fail — Subject Generalization and Distribution Shift.*

This is the scientific core. It answers the question Eli's JD names directly — *what can we conclude with
confidence, what remains uncertain* — for wrist-based stress detection. Five threads:

1. **Confounding is real and measured.** PPG quality (and thus HRV missingness) and motion both track the
   stress label. We formalize this before modelling so the model's number can be interpreted honestly.
2. **Leakage audit.** The same features and model report very different accuracy depending only on *how you
   split and scale*. We decompose the inflation into subject leakage and normalization leakage.
3. **Confounding probe.** Ablate motion (`acc_*`); train on motion alone. Is this a stress detector or a
   motion detector?
4. **LOSO with confidence intervals.** The honest generalization number, with per-subject spread and a
   bootstrap CI — a point estimate on 15 subjects would be dishonest.
5. **Calibration.** Are the predicted probabilities trustworthy, or only the labels?

Feature policy from notebook 02: model on `TRUSTED_FEATURES` only. `hr_std`/`hrv_*` excluded (E4 HRV
unreliable); `ppg_quality` is a **covariate**, never a feature (it's almost a label proxy).

> Input: `outputs/tables/wrist_features_w60_s60.csv`. sklearn + scipy only (no air-gapped installs).

## 1 · Load features & define the modelling matrix

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, brier_score_loss

RANDOM_STATE = 0
TAB = Path("outputs/tables"); FIG = Path("outputs/figures"); FIG.mkdir(parents=True, exist_ok=True)

feat = pd.read_csv(TAB / "wrist_features_w60_s60.csv")

# trusted feature list (from nb 02); fallback if the file isn't present
try:
    ft = pd.read_csv(TAB / "feature_trust.csv")
    TRUSTED = ft.loc[ft.trusted, "feature"].tolist()
except FileNotFoundError:
    TRUSTED = ["eda_mean","eda_std","eda_min","eda_max","eda_slope","eda_scr_count",
               "temp_mean","temp_std","temp_min","temp_max","temp_slope",
               "hr_mean","acc_std","acc_energy","acc_mad","acc_max"]
TRUSTED = [f for f in TRUSTED if f in feat.columns]
ACC_FEATS = [f for f in TRUSTED if f.startswith("acc_")]

X   = feat[TRUSTED].values
y   = feat["condition"].values           # 1=baseline 2=stress 3=amusement
grp = feat["subject"].values
print(f"{len(feat)} windows | {len(TRUSTED)} trusted features | {feat.subject.nunique()} subjects")
print("classes:", dict(zip(*np.unique(y, return_counts=True))))
print("ppg_quality and hr_std/hrv_* are NOT in X:", "ppg_quality" not in TRUSTED and "hr_std" not in TRUSTED)

## 2 · Confounding, measured before modelling

Two covariates that must not be mistaken for physiology. If they track the label, any model that indirectly
picks them up is inflated.

In [ ]:
order = [1, 2, 3]; names = {1:"baseline",2:"stress",3:"amusement"}

# (a) PPG quality by condition -- Kruskal-Wallis (non-parametric; quality is bounded/skewed)
groups_q = [feat.loc[feat.condition==c, "ppg_quality"].values for c in order]
H, p_q = stats.kruskal(*groups_q)
print("PPG quality by condition:")
for c in order:
    v = feat.loc[feat.condition==c, "ppg_quality"]
    print(f"  {names[c]:9s} median={v.median():.3f}  mean={v.mean():.3f}  n={len(v)}")
print(f"  Kruskal-Wallis H={H:.1f}, p={p_q:.2e}  -> quality {'IS' if p_q<0.05 else 'is not'} confounded with label\n")

# (b) HRV missingness by condition -- chi-square (is HRV missing-not-at-random w.r.t. label?)
feat["hrv_missing"] = feat["hrv_rmssd"].isna()
ct = pd.crosstab(feat["condition"].map(names), feat["hrv_missing"])
chi2, p_m, _, _ = stats.chi2_contingency(ct)
print("HRV missingness by condition (True = HRV unavailable):")
print(ct)
print(f"  chi2={chi2:.1f}, p={p_m:.2e}  -> HRV missingness {'IS' if p_m<0.05 else 'is not'} "
      f"confounded with label (MNAR) => do not impute HRV")

## 3 · Leakage audit

Identical features and model (`StandardScaler → LogisticRegression`). Only the **split** and **where the
scaler is fit** change. The inflation decomposes into two sources:

- **normalization leakage** = scaling fit on all data vs. fit per training fold
- **subject leakage** = random split (a subject's windows in both train and test) vs. LOSO

Given the baseline ICC of 0.85–0.95 from notebook 01, subject leakage should dominate — the model can
recognise the *person*.

In [ ]:
def macro_f1_cv(estimator, X, y, cv, groups=None):
    yp = cross_val_predict(estimator, X, y, cv=cv, groups=groups)
    return f1_score(y, yp, average="macro"), balanced_accuracy_score(y, yp)

pipe = make_pipeline(StandardScaler(),
                     LogisticRegression(max_iter=5000, class_weight="balanced"))
skf  = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
logo = LeaveOneGroupOut()

# (a) random CV + GLOBAL scaling (scale entire X up front -> leaks test stats into train)
Xg = StandardScaler().fit_transform(X)
f_a, b_a = macro_f1_cv(LogisticRegression(max_iter=5000, class_weight="balanced"), Xg, y, skf)
# (b) random CV + per-fold scaling (scaler inside pipeline) -- subject leakage remains
f_b, b_b = macro_f1_cv(pipe, X, y, skf)
# (c) LOSO + per-fold scaling -- honest
f_c, b_c = macro_f1_cv(pipe, X, y, logo, groups=grp)

audit = pd.DataFrame([
    ["naive",   "random 5-fold", "global",   f_a, b_a, "subject + normalization"],
    ["partial", "random 5-fold", "per-fold", f_b, b_b, "subject"],
    ["honest",  "LOSO",          "per-fold", f_c, b_c, "none"],
], columns=["pipeline","split","scaling","macro_F1","bal_acc","leak_sources"])
audit.to_csv(TAB/"leakage_audit.csv", index=False)
print(audit.to_string(index=False))
print(f"\n  normalization-leakage inflation (naive - partial): {f_a-f_b:+.3f} macro-F1")
print(f"  SUBJECT-leakage inflation      (partial - honest): {f_b-f_c:+.3f} macro-F1  <- the headline")
print(f"  total inflation                (naive  - honest): {f_a-f_c:+.3f} macro-F1")

ax = audit.plot.bar(x="pipeline", y="macro_F1", legend=False, figsize=(6,4),
                    color=["#C44E52","#DD8452","#4C72B0"])
ax.set_ylabel("macro-F1"); ax.set_title("Same model, same features — only the split/scaling differ")
for i,v in enumerate(audit.macro_F1): ax.text(i, v+.01, f"{v:.2f}", ha="center")
plt.tight_layout(); plt.savefig(FIG/"leakage_audit.png", dpi=140); plt.show()

### 3.1 · Is the gap model-specific? Repeat honest LOSO with a non-linear model
If a gradient-boosted tree shows the same honest-vs-leaked pattern, the leakage is a property of the
*validation design*, not of logistic regression.

In [ ]:
hgb = HistGradientBoostingClassifier(random_state=RANDOM_STATE, class_weight="balanced")
f_hgb_rand, _ = macro_f1_cv(hgb, X, y, skf)
f_hgb_loso, _ = macro_f1_cv(hgb, X, y, logo, groups=grp)
print(f"HistGradientBoosting  random-CV macro-F1 = {f_hgb_rand:.3f}")
print(f"HistGradientBoosting  LOSO      macro-F1 = {f_hgb_loso:.3f}")
print(f"  subject-leakage gap = {f_hgb_rand-f_hgb_loso:+.3f}  (compare LogReg {f_b-f_c:+.3f})")

## 4 · Confounding probe — stress detector or motion detector?

Three LOSO evaluations: full trusted set, motion removed (`acc_*` dropped), and **motion only**. If dropping
ACC barely changes accuracy *and* ACC-alone is near chance, motion isn't carrying the prediction. If ACC-alone
scores well above chance, the stress label has a motion signature the model can exploit — a confound to
disclose. For reference we also show what leaking `ppg_quality` in as a feature would do (it shouldn't be a
feature; this quantifies the confound's potential).

In [ ]:
def loso_f1(cols):
    return macro_f1_cv(pipe, feat[cols].values, y, logo, groups=grp)[0]

chance = f1_score(y, np.full_like(y, pd.Series(y).mode()[0]), average="macro")
f_full   = loso_f1(TRUSTED)
f_no_acc = loso_f1([c for c in TRUSTED if c not in ACC_FEATS])
f_acc    = loso_f1(ACC_FEATS)
f_leakq  = loso_f1(TRUSTED + ["ppg_quality"])      # DEMO of confound only -- never ship this

print(f"chance (majority-class macro-F1)     : {chance:.3f}")
print(f"LOSO full trusted set                : {f_full:.3f}")
print(f"LOSO without motion (acc_* dropped)  : {f_no_acc:.3f}   (delta {f_no_acc-f_full:+.3f})")
print(f"LOSO motion ONLY (acc_* )            : {f_acc:.3f}   (vs chance {f_acc-chance:+.3f})")
print(f"LOSO + ppg_quality leaked in         : {f_leakq:.3f}   (delta {f_leakq-f_full:+.3f}) <- why it's banned")
pd.DataFrame(dict(setting=["chance","full","no_ACC","ACC_only","+ppg_quality(leak)"],
                  macro_F1=[chance,f_full,f_no_acc,f_acc,f_leakq])
             ).to_csv(TAB/"confounding_probe.csv", index=False)

## 5 · LOSO generalization with confidence intervals

The honest headline number, but never as a bare point estimate on 15 subjects. We report the **per-subject**
macro-F1 distribution, a bootstrap CI over subjects, the confusion matrix, and the worst subjects — the
distribution shift the JD cares about lives in the spread, not the mean.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
per_subj, subj_ids = [], []
for tr, te in logo.split(X, y, grp):
    m = clone(pipe).fit(X[tr], y[tr])
    per_subj.append(f1_score(y[te], m.predict(X[te]), average="macro"))
    subj_ids.append(grp[te][0])
per_subj = np.array(per_subj)

boot = [rng.choice(per_subj, len(per_subj), replace=True).mean() for _ in range(5000)]
ci = np.percentile(boot, [2.5, 97.5])
print(f"LOSO macro-F1: mean={per_subj.mean():.3f}  sd={per_subj.std(ddof=1):.3f}  "
      f"95% CI [{ci[0]:.3f}, {ci[1]:.3f}]")
print(f"  range across subjects: {per_subj.min():.3f} ({subj_ids[per_subj.argmin()]}) "
      f"to {per_subj.max():.3f} ({subj_ids[per_subj.argmax()]})")
gap = f_b - per_subj.mean()
print(f"  generalization gap (random-CV - LOSO): {gap:+.3f} macro-F1")

pd.DataFrame(dict(subject=subj_ids, loso_macro_f1=per_subj)).to_csv(TAB/"loso_per_subject.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sidx = np.argsort(per_subj)
ax[0].bar(np.array(subj_ids)[sidx], per_subj[sidx], color="#4C72B0")
ax[0].axhline(per_subj.mean(), color="k", ls="--", lw=1, label=f"mean {per_subj.mean():.2f}")
ax[0].axhline(chance, color="0.6", ls=":", lw=1, label=f"chance {chance:.2f}")
ax[0].set_ylabel("LOSO macro-F1"); ax[0].set_title("Per-subject generalization (sorted)")
ax[0].tick_params(axis="x", rotation=90, labelsize=8); ax[0].legend(fontsize=8)

yp_loso = cross_val_predict(pipe, X, y, cv=logo, groups=grp)
cm = confusion_matrix(y, yp_loso, normalize="true")
im = ax[1].imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax[1].set_xticks(range(3)); ax[1].set_yticks(range(3))
ax[1].set_xticklabels([names[c] for c in order], rotation=20); ax[1].set_yticklabels([names[c] for c in order])
ax[1].set_xlabel("predicted"); ax[1].set_ylabel("true"); ax[1].set_title("LOSO confusion (row-normalised)")
for i in range(3):
    for j in range(3):
        ax[1].text(j, i, f"{cm[i,j]:.2f}", ha="center",
                   color="white" if cm[i,j]>0.5 else "black")
plt.tight_layout(); plt.savefig(FIG/"loso_generalization.png", dpi=140); plt.show()

## 6 · Calibration — are the probabilities trustworthy?

A model can be right on labels yet badly over-confident. We take LOSO out-of-fold probabilities, bin by
predicted confidence, and compare to observed accuracy (reliability), plus a multiclass Brier score. This is
the "uncertainty" half of the JD's question, and it's what a longitudinal health product actually needs.

In [ ]:
proba = cross_val_predict(pipe, X, y, cv=logo, groups=grp, method="predict_proba")
classes = np.unique(y)
conf = proba.max(axis=1)
pred = classes[proba.argmax(axis=1)]
correct = (pred == y).astype(float)

# multiclass Brier = mean sum_k (p_k - 1[y=k])^2
onehot = np.zeros_like(proba); 
for i,c in enumerate(classes): onehot[y==c, i] = 1
brier = float(np.mean(((proba - onehot)**2).sum(axis=1)))

bins = np.linspace(0,1,11); idx = np.digitize(conf, bins)-1
xs, ys, ns = [], [], []
for b in range(10):
    m = idx==b
    if m.sum()>0: xs.append(conf[m].mean()); ys.append(correct[m].mean()); ns.append(m.sum())
print(f"Multiclass Brier score (LOSO): {brier:.3f}   (0=perfect; lower is better)")
print(f"Mean confidence {conf.mean():.3f} vs overall accuracy {correct.mean():.3f} "
      f"-> {'over' if conf.mean()>correct.mean() else 'under'}-confident")

plt.figure(figsize=(5,5))
plt.plot([0,1],[0,1],"k--",lw=1,label="perfect")
plt.plot(xs, ys, "o-", color="#C44E52", label="model")
plt.xlabel("predicted confidence"); plt.ylabel("observed accuracy")
plt.title(f"LOSO reliability (Brier={brier:.3f})"); plt.legend(); plt.tight_layout()
plt.savefig(FIG/"calibration.png", dpi=140); plt.show()

## 7 · Findings & handoff

Fill in from your run — these bracketed numbers are your CV bullets and interview talking points.

- **Confounding is real:** PPG quality differs by condition (Kruskal-Wallis p ≈ `[__]`); HRV missingness is
  MNAR (chi-square p ≈ `[__]`). HRV excluded and not imputed; `ppg_quality` kept out of features.
- **Leakage inflates macro-F1 by `[__]`** (naive `[__]` → honest `[__]`), almost all of it **subject
  leakage**, exactly as the ICC predicted. Same pattern under a non-linear model → it's the design, not the model.
- **Confounding probe:** dropping motion changes macro-F1 by `[__]`; motion-alone scores `[__]` vs chance
  `[__]`. Interpretation: the model is `[mostly physiology / partly a motion detector]`.
- **Honest LOSO macro-F1 = `[__]` (95% CI `[__, __]`)**, per-subject range `[__]`–`[__]`; worst subject
  `[S__]`. The generalization gap is `[__]`.
- **Calibration:** Brier `[__]`, `[over/under]`-confident — matters for a health product that must know when
  it doesn't know.

**Next:**
- **`04_slurm_sweep`** — the full LOSO × seed × config sweep (incl. the overlapping-window leakage at
  shift = 0.25 s, and subject-normalized vs raw features) as a SLURM job array, with CIs across seeds.
- **`05_statistics`** — mixed-effects ANOVA + FDR + Simpson's paradox on `subject_condition_means.csv`,
  formalising why subject-normalization is required.

Upload the `outputs/tables/*.csv` and `outputs/figures/*.png` from this notebook.